# Session 0d - Visualization Reference

**Asynchronous · reference, not a lecture · exercised in Session 2**

---

## What this is, and why it is not a session

This course has no standalone visualization session, and that is deliberate.

A dedicated session becomes an API tour - "here is `hist`, here is `boxplot`" - and
students who "did visualization in week 3" reliably stop plotting by week 8, which
is exactly when residual plots, learning curves and calibration curves start to
matter most. So the mechanics live here, in a reference you keep open, and the
*reasoning* is taught in Session 2 and then demanded in every session afterwards.

The organising principle is the one that matters:

> **You do not choose a chart. You choose a question, and the question chooses the
> chart.**

This notebook is therefore indexed by question, not by chart type. Use §1 to find
your question, then read the recipe.

## Learning objectives

1. Given a question about data, name an appropriate chart and justify it.
2. Build any of those charts with matplotlib's object API.
3. State what a given chart cannot tell you.
4. Recognise the four ways a chart misleads.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# A small synthetic dataset shaped like the course project, so the examples are
# recognisable without needing the real file.
n = 600
demo = pd.DataFrame({
    "price": np.exp(rng.normal(5.0, 0.6, n)).round(0),
    "capacity": rng.integers(1, 9, n),
    "reviews": rng.poisson(18, n),
    "rating": np.clip(rng.normal(4.6, 0.35, n), 1, 5).round(2),
    "room_type": rng.choice(["Entire home", "Private room", "Shared room"],
                            n, p=[0.66, 0.30, 0.04]),
    "district": rng.choice(["Eixample", "Ciutat Vella", "Gracia", "Sants"],
                           n, p=[0.4, 0.25, 0.2, 0.15]),
})
demo["log_price"] = np.log1p(demo["price"])
print(demo.head(3).to_string(index=False))
print(f"\n{len(demo)} rows")

## §1 - The index: find your question

| Your question | Chart | Section |
|---|---|---|
| How is one numeric variable distributed? | histogram, or ECDF | §2 |
| Is it skewed? Are there outliers? | histogram + boxplot | §2 |
| How often does each category occur? | bar chart (**not** a pie chart) | §2 |
| Do two numeric variables move together? | scatter plot | §3 |
| …with thousands of points overlapping? | hexbin, or scatter with alpha | §3 |
| Does a numeric variable differ across groups? | boxplot, or violin | §3 |
| Does the *relationship* differ across groups? | scatter with `hue`, or facets | §4 |
| Which variables are correlated with which? | correlation heatmap | §4 |
| How does a model's error behave? | residuals vs predicted | §5 |
| Is my model overfitting? | train/validation curves | §5 |
| Which class does my classifier confuse? | confusion matrix | §5 |
| How good is my ranking? | ROC or precision-recall curve | §5 |
| Is my probability trustworthy? | calibration curve | §5 |

Two rules before any of the recipes:

**Use the object API.** `fig, ax = plt.subplots()` then `ax.plot(...)`. The
alternative - `plt.plot()` on an implicit "current axes" - works for one throwaway
chart and becomes unmanageable the moment you want two panels.

**Label everything.** An unlabelled axis is not a chart, it is a decoration. Every
example below sets a title and axis labels, and the title states the *question*.

## §2 - One variable

### Numeric: histogram, and why you usually also want the log

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

axes[0].hist(demo["price"], bins=40, color="steelblue", edgecolor="white")
axes[0].set(title=f"Q: how are prices distributed?  (skew {demo['price'].skew():.2f})",
            xlabel="price (EUR)", ylabel="listings")

axes[1].hist(demo["log_price"], bins=40, color="darkorange", edgecolor="white")
axes[1].set(title=f"the same data, logged  (skew {demo['log_price'].skew():.2f})",
            xlabel="log1p(price)")

# An ECDF makes quantiles readable directly, and needs no bin-width choice.
ordered = np.sort(demo["price"])
axes[2].plot(ordered, np.arange(1, len(ordered) + 1) / len(ordered), color="seagreen")
axes[2].set(title="Q: what fraction are under EUR 200?", xlabel="price (EUR)",
            ylabel="cumulative fraction")
axes[2].axvline(200, color="grey", ls=":")

plt.tight_layout()
plt.show()

**The bin width is a decision, not a default.** Too few bins hides structure; too
many turns the chart into noise. If a conclusion changes when you change `bins`, it
was not a conclusion.

**What a histogram cannot tell you:** whether the long right tail is genuine luxury
or data-entry error. It shows that extremes *exist*; only looking at the rows tells
you what they are.

### Categorical: a bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

counts = demo["room_type"].value_counts()
axes[0].barh(counts.index[::-1], counts.to_numpy()[::-1], color="steelblue")
axes[0].set(title="Q: what is the composition of the market?", xlabel="listings")

# Sorted, horizontal, with readable labels - the default vertical bar chart with
# rotated text is harder to read for anything but very short category names.
by_district = demo["district"].value_counts().sort_values()
axes[1].barh(by_district.index, by_district.to_numpy(), color="cadetblue")
axes[1].set(title="Q: where are the listings?", xlabel="listings")

plt.tight_layout()
plt.show()

**Do not use a pie chart.** Humans compare lengths far more accurately than angles
or areas, so a sorted bar chart is strictly easier to read. The only case for a pie
is two or three categories summing to a meaningful whole, and a bar chart is still
fine there.

## §3 - Two variables

### Numeric vs numeric: scatter, and what to do about overplotting

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].scatter(demo["capacity"], demo["log_price"], s=10)
axes[0].set(title="Q: does price rise with capacity?  (raw)",
            xlabel="capacity", ylabel="log1p(price)")

# Two fixes for overlap: transparency, and jitter for a discrete axis.
jitter = demo["capacity"] + rng.normal(0, 0.12, len(demo))
axes[1].scatter(jitter, demo["log_price"], s=10, alpha=0.25)
axes[1].set(title="alpha + jitter: the density becomes visible", xlabel="capacity")

hb = axes[2].hexbin(demo["reviews"], demo["log_price"], gridsize=25, cmap="Blues")
axes[2].set(title="hexbin: for thousands of points", xlabel="reviews")
plt.colorbar(hb, ax=axes[2], label="count")

plt.tight_layout()
plt.show()

The left panel is the trap. With a discrete x-axis and 600 points, hundreds sit
exactly on top of each other and the chart shows you the *range* of prices at each
capacity while hiding where the mass is. The middle panel shows the same data
honestly.

> **Overplotting does not look like a problem.** It looks like a chart. Always ask
> how many points are hidden.

### Numeric vs categorical: boxplot and violin

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

sns.boxplot(data=demo, x="room_type", y="log_price", ax=axes[0],
            order=["Entire home", "Private room", "Shared room"])
axes[0].set(title="Q: how does price differ by room type?", xlabel="", ylabel="log1p(price)")

sns.violinplot(data=demo, x="room_type", y="log_price", ax=axes[1],
               order=["Entire home", "Private room", "Shared room"], cut=0)
axes[1].set(title="violin: the same, plus the shape", xlabel="", ylabel="")

# Always show the n. A confident-looking box over 24 observations is not a finding.
for i, room in enumerate(["Entire home", "Private room", "Shared room"]):
    axes[0].text(i, demo["log_price"].min(), f"n={(demo['room_type'] == room).sum()}",
                 ha="center", va="bottom", fontsize=8, color="dimgrey")

plt.tight_layout()
plt.show()

**Annotate the group sizes.** Seaborn will draw an equally confident box over 24
observations and over 400. The `n=` labels above take one line and prevent a whole
class of overclaiming - this is exactly the mistake Session 2 flags with the 119
shared rooms in the real dataset.

**What a boxplot cannot tell you:** whether room type *causes* the price difference.
Entire homes are also larger and differently located. It is a comparison, not an
effect.

## §4 - Three or more variables

The question changes from "do these two relate?" to "**does that relationship depend
on something else?**" - which is a question about interaction.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.lineplot(data=demo, x="capacity", y="log_price", hue="room_type", ax=axes[0],
             marker="o", errorbar=("ci", 95))
axes[0].set(title="Q: does the capacity-price slope depend on room type?",
            ylabel="log1p(price)")

numeric = demo[["log_price", "capacity", "reviews", "rating"]]
corr = numeric.corr()
im = axes[1].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
axes[1].set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        axes[1].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                     fontsize=9)
axes[1].set(title="Q: which variables move together?")
plt.colorbar(im, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

### Three things a correlation heatmap cannot tell you

1. **Anything non-linear.** Pearson correlation near zero is entirely consistent
   with a strong curved relationship. Session 3 has the definitive example: a column
   correlating −0.05 with the target that was one of the largest leaks in the
   dataset, because the relationship was a *ratio*.
2. **Direction of causation.** Never.
3. **Whether a third variable explains both.** Almost always the interesting case.

Use a heatmap to generate questions, never to answer them.

### Facets, when `hue` gets crowded

In [ ]:
grid = sns.FacetGrid(demo, col="district", col_wrap=2, height=2.4, aspect=1.5)
grid.map_dataframe(sns.scatterplot, x="capacity", y="log_price", s=12, alpha=0.5)
grid.set_titles("{col_name}")
grid.figure.suptitle("Q: does the pattern hold in every district?", y=1.02)
plt.show()

**Facets beat `hue` past about four groups.** Overlapping colours become unreadable,
and the eye is much better at comparing small multiples side by side. Keep the axes
shared (the default) so the panels are actually comparable.

## §5 - Model diagnostics

These are the charts you will be *required* to produce from Session 4 onwards, and
the ones most often skipped. Each answers a question a score cannot.

In [ ]:
# Stand-in model output, so the recipes are self-contained.
truth = demo["log_price"].to_numpy()
predicted = truth + rng.normal(0, 0.35, len(truth))
residual = truth - predicted

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

axes[0].scatter(predicted, residual, s=8, alpha=0.3)
axes[0].axhline(0, color="black", lw=1)
axes[0].set(title="Q: is the error patterned?", xlabel="predicted", ylabel="residual")

axes[1].hist(residual, bins=40, color="darkorange")
axes[1].set(title=f"Q: are errors symmetric?  (skew {pd.Series(residual).skew():+.2f})",
            xlabel="residual")

epochs = np.arange(1, 61)
train_curve = 0.9 * np.exp(-epochs / 12) + 0.06
val_curve = 0.9 * np.exp(-epochs / 12) + 0.06 + 0.0016 * np.clip(epochs - 22, 0, None)
axes[2].plot(epochs, train_curve, label="training")
axes[2].plot(epochs, val_curve, label="validation")
axes[2].axvline(epochs[int(np.argmin(val_curve))], color="grey", ls=":",
                label="best epoch")
axes[2].set(title="Q: am I overfitting?", xlabel="epoch", ylabel="loss")
axes[2].legend()

plt.tight_layout()
plt.show()

The third panel is the single most important chart in Block 3. **Plot validation
from epoch 1, not at the end** - the gap opening is the whole signal, and you cannot
see it retrospectively.

### Classification diagnostics

In [ ]:
from sklearn.metrics import (ConfusionMatrixDisplay, precision_recall_curve,
                             roc_curve)

# Stand-in scores for a 2/3-positive problem, like the course's licensing target.
label = rng.binomial(1, 0.667, 1200)
score = np.clip(0.5 + 0.28 * (label - 0.5) + rng.normal(0, 0.2, 1200), 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))

ConfusionMatrixDisplay.from_predictions(label, (score >= 0.5).astype(int),
                                        ax=axes[0], colorbar=False)
axes[0].set(title="Q: which errors am I making?")

fpr, tpr, _ = roc_curve(label, score)
axes[1].plot(fpr, tpr)
axes[1].plot([0, 1], [0, 1], "k:", label="chance")
axes[1].set(title="Q: how well do I rank?", xlabel="false positive rate",
            ylabel="true positive rate")
axes[1].legend()

precision, recall, _ = precision_recall_curve(label, score)
axes[2].plot(recall, precision)
axes[2].axhline(label.mean(), color="grey", ls=":",
                label=f"floor = base rate {label.mean():.2f}")
axes[2].set(title="Q: precision at each recall?", xlabel="recall", ylabel="precision")
axes[2].legend()

plt.tight_layout()
plt.show()

**Always draw the PR curve's floor.** It is the base rate, not zero - the dotted
line above. A precision-recall curve without it looks far better than it is, and
Session 5 shows a do-nothing classifier scoring PR-AUC 0.667 on this course's data.

## §6 - Practical recipes

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot([1, 2, 3], [1, 4, 9])

# The six things worth remembering:
ax.set_title("state the question in the title")     # not "Figure 1"
ax.set_xlabel("x, with units")
ax.set_ylabel("y, with units")
ax.set_yscale("linear")                              # "log" for skewed quantities
ax.grid(True, alpha=0.3)                             # subtle, behind the data
fig.tight_layout()                                   # stops labels being clipped
plt.show()

print("""Saving for a report:
    fig.savefig("figure.png", dpi=200, bbox_inches="tight")

Two panels sharing a y-axis, so they are comparable:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

Ordering a categorical axis deliberately, instead of alphabetically:
    sns.boxplot(..., order=["Entire home", "Private room", "Shared room"])

Colour: use a SEQUENTIAL map for magnitude ("Blues"), DIVERGING for signed
quantities around a meaningful centre ("RdBu_r", as in the correlation heatmap),
and QUALITATIVE for unordered categories ("tab10"). Using a diverging map for an
unsigned quantity invents a midpoint that does not exist.""")

## §7 - The four ways a chart misleads

This section is the one with real content. Read it properly.

### 1. Hidden sample size

A boxplot over 24 observations looks exactly as authoritative as one over 4,000.
Seaborn will happily draw a confident 95% interval through almost nothing.
**Annotate n.**

### 2. Overplotting

Thousands of coincident points look like a sparse cloud. You see the *extent* of the
data and not its *density*, which usually reverses the impression. **Use alpha,
jitter, or hexbin, and say how many points there are.**

### 3. Axis choices that do the arguing

A truncated y-axis magnifies a trivial difference; a linear axis on a
nine-orders-of-magnitude quantity hides everything below the maximum. Neither is
dishonest by itself, and both are choices you are responsible for. **If the
conclusion changes when you change the axis, the axis was the conclusion.**

### 4. Reading causation off a shape

The most common and most costly. A downward scatter is compatible with: X causing Y,
Y causing X, a third variable causing both, and a selection effect you introduced
yourself while cleaning. Session 2 works through all four on the real data.

> For every chart you make, answer both questions:
> **what does this show?** and **what does this specifically fail to show?**
>
> The second answer is what distinguishes an analyst from someone who owns a
> plotting library.

## §8 - Exercises

Do these before Session 2. Each needs one chart and two sentences.

In [ ]:
# TODO: For each question, produce a chart and write (a) what it shows and (b) one
# thing it cannot show. Use the `demo` frame.
#
# 1. Are high-capacity listings more expensive, and does that depend on room type?
# 2. Which district has the widest spread of prices?
# 3. Is `rating` associated with `price`? Be careful with this one.
# 4. Pick any chart above and deliberately make it mislead, using one of the four
#    mechanisms from §7. Say which mechanism you used.